## LIBRARIES AND DECLARATION

In [ ]:
import pandas as pd
import json
import re
from IPython.display import FileLink

In [ ]:
df1 = pd.read_csv("/kaggle/input/datasets/chauanhh/step2-logic1/step2_logic2_qwen3.7-max_1.csv")
df2 = pd.read_csv("/kaggle/input/datasets/chauanhh/step2-logic1/step2_logic2_qwen3.7-max_2.csv")
df3 = pd.read_csv("/kaggle/input/datasets/chauanhh/step2-logic1/step2_logic2_qwen3.7-max_3.csv")

df = pd.concat([df1, df2, df3], ignore_index=False)
print(df.shape[0])

## LABEL CLASSIFICATION

- Label 1: 3/3 judge attempts have `pred_status` matching `gt_status`
- Label 2: 1/3 or 2/3 judge attempts have `pred_status` matching `gt_status`
- Label 3: 0/3 judge attempts have `pred_status` matching `gt_status`

In [ ]:
# So sánh gt_status vs pred_status
df["match"] = df.apply(
    lambda r: "sim" if r["gt_status"] == r["pred_status"] else "diff",
    axis=1
)

# Group theo id, đếm sim - diff trong mỗi nhóm
group_stats = (
    df.groupby("id")["match"]
    .value_counts()
    .unstack(fill_value=0)
    .rename(columns={"sim": "n_sim", "diff": "n_diff"})
    .reset_index()
)

# Đảm bảo cả hai cột luôn tồn tại dù tất cả là sim hoặc diff
for col in ("n_sim", "n_diff"):
    if col not in group_stats.columns:
        group_stats[col] = 0

# Gán label cho mỗi id
def assign_label(row):
    s, d = row["n_sim"], row["n_diff"]
    total = s + d
    if total != 3:
        # Nếu nhóm không có đúng 3 dòng, trả về None để dễ debug
        return None
    if d == 3:          # 3 diff
        return 3
    if s == 3:          # 3 sim
        return 1
    # 2 sim 1 diff  →  label 2
    # 1 sim 2 diff  →  label 2
    return 2

group_stats["label"] = group_stats.apply(assign_label, axis=1)

df = df.merge(group_stats[["id", "label"]], on="id", how="left")

print(df[["id", "gt_status", "pred_status", "match", "label"]].head(20))
print(df.drop_duplicates("id")["label"].value_counts().sort_index())
df_stage1 = df.drop_duplicates("id").reset_index(drop=True)
df_stage1.to_csv("step1_classify1_logic1.csv", index=False)

In [ ]:
FileLink(r'step1_classify1_logic1.csv')